This is to find best parm based on TimeNet

#**Pre-request**

##Mount google drive


In [1]:
### **Mount** Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


##Install pakages


In [2]:
#Install pakages
project_path = "/content/drive/MyDrive/Sem-6/coding/github/fraud_detection/"
!cat "{project_path}requirement/Install/NASEnhancedPretraindMLModleAdvance.txt"
!pip install  -r "{project_path}requirement/Install/NASEnhancedPretraindMLModleAdvance.txt" --no-cache-dir
%cd $project_path





torch
transformers
huggingface_hub
datasets
timm
patool
sktime
reformer_pytorch
optuna
ptflopsRequirement already satisfied: torch in /usr/local/lib/python3.12/dist-packages (from -r /content/drive/MyDrive/Sem-6/coding/github/fraud_detection/requirement/Install/NASEnhancedPretraindMLModleAdvance.txt (line 1)) (2.11.0+cu128)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.4/101.4 kB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.5/37.5 MB 377.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 388.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 371.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.3/165.3 kB 353.2 MB/s eta 0:00:00
/content/drive/MyDrive/Sem-6/coding/github/fraud_detection


##Import  libs

In [3]:
# =====================================================
# 📦 Standard Library
# =====================================================
import os
import sys
import time
import logging
import hashlib
import shutil
import subprocess
import warnings
from datetime import datetime

# Start timer
start_time = time.time()

# =====================================================
# 🧮 Data & Visualization
# =====================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml

# Pandas display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

# =====================================================
# ⚙️ Machine Learning - Scikit-learn
# =====================================================
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, RobustScaler, StandardScaler
from sklearn.utils import class_weight
from sklearn.covariance import EmpiricalCovariance
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)

# =====================================================
# 🌲 XGBoost
# =====================================================
from xgboost import XGBClassifier
import joblib

# =====================================================
# 🔥 Deep Learning - PyTorch
# =====================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from torch.cuda.amp import autocast

# =====================================================
# 🤖 Deep Learning - TensorFlow / Keras
# =====================================================
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import LSTM, Dense, Masking, Dropout, Layer
from tensorflow.keras.optimizers import Adam

# =====================================================
# 🤗 Transformers & Time Series
# =====================================================
from transformers import AutoModel
from sktime.datasets import load_from_tsfile_to_dataframe
# from mamba_ssm import Mamba  # Uncomment if needed

# =====================================================
# 🧠 Explainability
# =====================================================
import shap

# =====================================================
# 📊 Google Colab Specific
# =====================================================
from google.colab import data_table
data_table.enable_dataframe_formatter()
try:
    from google.colab import data_table
    data_table.enable_dataframe_formatter()
    data_table.DataTable.max_columns = 50
    data_table.DataTable.max_rows = 150
except ImportError:
    pass

from tqdm import tqdm

print("✅ All imports loaded successfully!")

✅ All imports loaded successfully!


##Confirmation setup

In [4]:
# =====================================================
# 🎲 Random Seeds (Reproducibility)
# =====================================================
!nvidia-smi                # confirm GPU
!pip show torch  # confirm versions
torch.manual_seed(42)
np.random.seed(42)

Thu Jul  9 01:43:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   35C    P0             56W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## Enable Config

In [5]:

logger = logging.getLogger(__name__)

def load_config(config_path="configs/baseline.yaml"):
    """Load YAML config file and expand ${root_path} placeholders."""
    with open(config_path, "r") as f:
        config = yaml.safe_load(f)

    logger.info(f"✅ Loaded config from {config_path}")

    # --- Expand ${root_path} placeholders ---
    root = config.get("root_path", "")

    def expand_paths(obj):
        if isinstance(obj, dict):
            return {k: expand_paths(v) for k, v in obj.items()}
        elif isinstance(obj, list):
            return [expand_paths(i) for i in obj]
        elif isinstance(obj, str) and "${root_path}" in obj:
            return obj.replace("${root_path}", root)
        else:
            return obj

    config = expand_paths(config)
    return config
config = load_config(os.path.join(project_path, "configs", "baseline.yaml"))


## Set Variables

In [6]:


#limit = config['ML']['limit']
n_trials=100
#
# ==========================================================
# UNIFIED HYPERPARAMETERS (Match TimesNet NAS Best)
# ==========================================================
# ==========================================================
# 🔧 UNIFIED HYPERPARAMETERS (Match TimesNet NAS Best)
# ==========================================================

# ----------------------------------------------------------
# 📏 Sequence Settings
# ----------------------------------------------------------
max_seq_len = 6                  # Maximum sequence length
recent_mode = False               # False → oldest mode, True → recent-window mode
epochs=20
# ----------------------------------------------------------
# 🏗️ Model Architecture (Shared across LSTM/Transformer/TimesNet)
# ----------------------------------------------------------

# ----------------------------------------------------------
# 📊 Evaluation Settings
# ----------------------------------------------------------
threshold = 0.5                   # Default classification threshold
opt_metric = "f1"                 # Optimization metric for model selection
early_stop_metric = 'val_accuracy'# Metric for early stopping
correlation_threshold = 0.85      # Feature correlation threshold
best_threshold = 0.5              # Best threshold (general)


# ----------------------------------------------------------
# 💾 Paths
# ----------------------------------------------------------
model_path = config['ML']['models']

# ==========================================================
# ✅ Configuration Summary
# ==========================================================
print("="*60)
print("📋 CONFIGURATION SUMMARY")
print("="*60)
print(f"  Sequence length:  {max_seq_len}")
print(f"  Mode:             {'Recent' if recent_mode else 'Oldest'}")
print(f"  Threshold:        {threshold}")
print(f"  Model path:       {model_path}")
print("="*60)

# Global unified results table for all models
results_table = pd.DataFrame(columns=["Round", "AUC", "Recall", "F1", "Model"])
summary = pd.DataFrame(
    columns=[
        "Model",
        "AUC",
        "Recall",
        "Precision",
        "F1",
        "threshold"
    ]
)


📋 CONFIGURATION SUMMARY
  Sequence length:  6
  Mode:             Oldest
  Threshold:        0.5
  Model path:       /content/drive/MyDrive/Sem-6/coding/github/fraud_detection/models/


##Split users level

In [7]:

# user_path = config['ML']['Events']['base_path'] + config['ML']['Events']['files']['user']
# df_user = pd.read_csv(user_path)
# print(f"✅ Loaded transactional user dataset: {df_user.shape}")



# # Aggregate to one row per user (max label = 1 if any fraud)
# user_labels = df_user.groupby("phone_no_m")["label"].max()
# print(f"👥 Unique users for splitting: {len(user_labels)}")

# # ==============================================================
# # 2️⃣ Create user-level split (stratified, no leakage)
# # ==============================================================

# fraud_users = user_labels[user_labels == 1].index
# normal_users = user_labels[user_labels == 0].index

# fraud_train, fraud_test = train_test_split(fraud_users, test_size=0.2, random_state=42)
# normal_train, normal_test = train_test_split(normal_users, test_size=0.2, random_state=42)

# train_users = set(fraud_train) | set(normal_train)
# test_users  = set(fraud_test)  | set(normal_test)

# # ==============================================================
# # 3️⃣ Save unified split (shared across LSTM / RF / XGB)
# # ==============================================================

# split_dir = "splits/shared_user_split_v1"
# os.makedirs(split_dir, exist_ok=True)

# train_split_file = f"{split_dir}/train_users.csv"
# test_split_file  = f"{split_dir}/test_users.csv"
# val_split_file   = f"{split_dir}/val_users.csv"

# pd.DataFrame({"phone_no_m": sorted(train_users)}).to_csv(train_split_file, index=False)
# pd.DataFrame({"phone_no_m": sorted(test_users)}).to_csv(test_split_file, index=False)
# ==============================================================
# 4️⃣ Summary
# ==============================================================

# print("\n👥 Users Summary:")
# print(f"   Total : {len(user_labels):,}")
# print(f"   Fraud : {len(fraud_users):,} ({len(fraud_users)/len(user_labels)*100:.2f}%)")
# print(f"   Normal: {len(normal_users):,} ({len(normal_users)/len(user_labels)*100:.2f}%)")

# print("\n📂 Split saved to /splits/:")
# print(f"   Train users: {len(train_users)}")
# print(f"   Test  users: {len(test_users)}")
# print(f"   Fraud ratio train: {len(fraud_train)/len(train_users)*100:.2f}%")
# print(f"   Fraud ratio test : {len(fraud_test)/len(test_users)*100:.2f}%")


## Helpers

### evaluate_global

In [8]:
def evaluate_global(model, X_test, y_test, model_name="Model", threshold=0.5):
    """
    Generic evaluator for both classic ML models and neural networks.
    """
    print(f"\n📊 Evaluation threshold is: {threshold}")

    # ---- Predict probabilities ----
    if hasattr(model, "predict_proba"):
        # For sklearn-style models
        y_pred_prob = model.predict_proba(X_test)[:, 1]
    else:
        # For neural nets (e.g., Keras)
        preds = model.predict(X_test)
        if preds.shape[-1] == 2:
            # 2-class softmax output
            y_pred_prob = preds[:, 1]
        else:
            # Single sigmoid output
            y_pred_prob = preds.ravel()

    # ... rest of function unchanged
    # ---- Predict classes ----
    y_pred = (y_pred_prob > threshold).astype(int)

    # ---- Metrics ----
    auc = roc_auc_score(y_test, y_pred_prob)
    #recall = recall_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred, zero_division=0)

    precision = precision_score(y_test, y_pred, zero_division=0)
    #f1 = f1_score(y_test, y_pred)
    f1     = f1_score(y_test, y_pred, zero_division=0)

    report = classification_report(y_test, y_pred, digits=4)
    cm = confusion_matrix(y_test, y_pred)

    # ---- Display ----
    print(f"\n📊 Classification Report — {model_name}")
    print(report)
    print(f"AUC: {auc:.4f} | Recall: {recall:.4f} | Precision: {precision:.4f} | F1: {f1:.4f}")

    # ---- Confusion Matrix ----
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Normal (0)", "Fraud (1)"])
    disp.plot(cmap="Blues")
    plt.title(f"Confusion Matrix — {model_name}")
    plt.grid(False)
    plt.show()

    # ---- Summary Dictionary ----
    return {
        "Model": model_name,
        "AUC": auc,
        "Recall": recall,
        "Precision": precision,
        "F1": f1,
        "threshold": threshold
    }



### append_to_summary

In [9]:

def append_to_summary(summary, model_name, results):
    """
    Appends or updates the summary table with model results.
    Works with both capitalized and lowercase keys automatically.
    """
    # ✅ Create summary DataFrame if missing
    if summary is None or not isinstance(summary, pd.DataFrame):
          summary = pd.DataFrame(columns=["Model", "AUC", "Recall", "Precision", "F1", "Threshold"])
    # Ensure "Model" column exists (prevents KeyError)
    if "Model" not in summary.columns:
        summary = pd.DataFrame(columns=["Model", "AUC", "Recall", "Precision", "F1", "Threshold"])

    # ✅ Normalize key names to lowercase
    results = {k.lower(): v for k, v in results.items()}

    # ✅ Remove any existing row for the same model
    summary = summary[summary["Model"] != model_name]

    # ✅ Add new row (robust to missing values)
    row = {
        "Model": model_name,
        "AUC": round(results.get("auc", np.nan), 4) if not pd.isna(results.get("auc", np.nan)) else np.nan,
        "Recall": round(results.get("recall", np.nan), 4) if not pd.isna(results.get("recall", np.nan)) else np.nan,
        "Precision": round(results.get("precision", np.nan), 4) if not pd.isna(results.get("precision", np.nan)) else np.nan,
        "F1": round(results.get("f1", np.nan), 4) if not pd.isna(results.get("f1", np.nan)) else np.nan,
        "Threshold": round(results.get("threshold", np.nan), 4) if not pd.isna(results.get("threshold", np.nan)) else np.nan
    }


    # ✅ Append and reindex
    summary = pd.concat([summary, pd.DataFrame([row])], ignore_index=True)
    summary = summary.reindex(columns=["Model", "AUC", "Recall", "Precision", "F1", "Threshold"])
    return summary


### find_best_threshold

In [10]:
def find_best_threshold(y_true, probs, low=0.2, high=0.8, n=61):
    best_f1 = -1.0
    best_thr = 0.5
    for thr in np.linspace(low, high, n):
        preds = (probs >= thr).astype(int)
        f1 = f1_score(y_true, preds, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_thr = thr
    return best_thr, best_f1


def get_timesnet_val_threshold(results_dir):
    val_prob_path = os.path.join(results_dir, "val_prob.npy")
    val_true_path = os.path.join(results_dir, "val_true.npy")

    if not os.path.exists(val_prob_path) or not os.path.exists(val_true_path):
        raise FileNotFoundError(
            f"Missing validation probability files in {results_dir}. "
            f"Expected val_prob.npy and val_true.npy"
        )

    val_probs = np.load(val_prob_path)
    val_true = np.load(val_true_path)

    best_thr, best_f1 = find_best_threshold(val_true, val_probs)
    return best_thr, best_f1

###Drop and select features

In [11]:
def prepare_features(df):
    """
    Selects only the explicitly defined features for model training.
    You control which features are used by editing 'selected_features' below.
    """

    # --- Define selected features manually ---
    selected_features = [
        "window_size", "voc_total_calls", "voc_unique_contacts", "voc_total_duration",
       "voc_avg_duration", "voc_max_duration", "voc_std_duration", "voc_active_days",
       "voc_active_hours", "sms_total_msgs", "sms_unique_contacts", "sms_active_hours",
       "sms_calltype_ratio", "app_months_active", "app_total_flow", "app_avg_flow",
       "app_std_flow", "app_unique_apps_mean", "app_unique_apps_max", "user_months_active",
        "arpu_mean", "arpu_std", "arpu_max", "idcard_cnt", "snapshot_round"
   ]
  #  selected_features = [
   #     "voc_total_calls", "voc_unique_contacts", "voc_total_duration",
    #   "voc_avg_duration", "voc_max_duration", "voc_std_duration", "voc_active_days",
     # "voc_active_hours", "sms_total_msgs", "sms_unique_contacts", "sms_active_hours",
     #"sms_calltype_ratio", "idcard_cnt"
    #]
   # selected_features = [
    #    "voc_active_days",
    #"voc_active_hours",
    #"voc_unique_contacts",
    #"sms_calltype_ratio",
    #"sms_active_hours" ]


    # ✅ You can manually remove or comment out features here
    # For example:
    # selected_features = [f for f in selected_features if not (f.startswith("app_") or f.startswith("arpu_"))]

    # --- Keep only existing columns ---
    available = [f for f in selected_features if f in df.columns]
    missing = [f for f in selected_features if f not in df.columns]

    X = df[available].copy()

    #print(f"\n📊 Final features used ({len(available)}): {available}")
    if missing:
        print(f"⚠️ Missing columns not found in data: {missing}")

    return X


### Compare

In [12]:

def plot_progressive_results(
    results_table,
    metrics=("AUC", "Recall", "F1"),
    round_col=None,
    figsize=(14, 6)
):
    """
    Plot progressive evaluation metrics per round for multiple models.

    Parameters
    ----------
    results_table : pd.DataFrame
        Must contain columns: Model, metrics, and either Round or input_size
    metrics : tuple
        Metrics to plot (default: AUC, Recall, F1)
    round_col : str or None
        Column name for x-axis. If None, auto-detects.
    figsize : tuple
        Figure size for plots
    """

    # --------------------------------------------------
    # Auto-detect round column
    # --------------------------------------------------
    if round_col is None:
        if "Round" in results_table.columns:
            round_col = "Round"
        elif "input_size" in results_table.columns:
            round_col = "input_size"
        else:
            raise ValueError("No Round or input_size column found.")

    # --------------------------------------------------
    # Sort results (important for correct curves)
    # --------------------------------------------------
    results_table = results_table.sort_values(
        by=[round_col, "Model"],
        ascending=True
    ).reset_index(drop=True)


    # --------------------------------------------------
    # Plot each metric
    # --------------------------------------------------
    for metric in metrics:

        plt.figure(figsize=figsize)

        for model in results_table["Model"].unique():
            subset = (
                results_table[results_table["Model"] == model]
                .sort_values(by=round_col)
            )

            plt.plot(
                subset[round_col],
                subset[metric],
                marker="o",
                markersize=6,
                linewidth=2,
                label=model,
                alpha=0.85
            )

        plt.title(f"{metric} per {round_col}", fontsize=18)
        plt.xlabel(round_col, fontsize=14)
        plt.ylabel(metric, fontsize=14)
        plt.grid(True, linestyle="--", alpha=0.4)

        # Legend outside
        plt.legend(
            loc="upper center",
            bbox_to_anchor=(0.5, -0.12),
            ncol=4,
            fontsize=10
        )

        plt.tight_layout(rect=[0, 0.1, 1, 1])
        plt.show()
    display(results_table)

    return results_table


###get_key_rounds

In [13]:
# ============================================================
# 🔬 SCIENTIFIC KEY ROUNDS SELECTION
# ============================================================

def get_key_rounds(max_seq_len, method="linear", n_points=5):
    """
    Generate scientifically meaningful evaluation checkpoints.

    Args:
        max_seq_len: Maximum sequence length (16, 100, 300, etc.)
        method: Selection strategy
            - "linear": Equal spacing (1, 25%, 50%, 75%, 100%)
            - "logarithmic": More points early (where changes happen fast)
            - "sqrt": Square root spacing (balanced)
            - "percentile": Fixed percentages
        n_points: Number of evaluation points (default 5)

    Returns:
        List of round numbers to evaluate
    """

    if max_seq_len <= n_points:
        # If sequence is short, evaluate all rounds
        return list(range(1, max_seq_len + 1))

    if method == "linear":
        # Equal spacing: 1, 25%, 50%, 75%, 100%
        rounds = np.linspace(1, max_seq_len, n_points)

    elif method == "logarithmic":
        # More points early (fraud detection often shows early signal)
        # Log scale: 1, 2, 4, 8, 16 style
        rounds = np.logspace(0, np.log10(max_seq_len), n_points)

    elif method == "sqrt":
        # Square root spacing (balanced between linear and log)
        rounds = np.square(np.linspace(1, np.sqrt(max_seq_len), n_points))

    elif method == "percentile":
        # Fixed percentages: 1st event, 10%, 25%, 50%, 75%, 100%
        percentages = [0, 0.1, 0.25, 0.5, 0.75, 1.0]
        rounds = [max(1, int(p * max_seq_len)) for p in percentages]
        return sorted(set(rounds))  # Remove duplicates

    elif method == "early_focus":
        # Focus on early detection (more points in first half)
        # Useful for fraud detection where early signal matters
        early = np.linspace(1, max_seq_len * 0.5, n_points - 2)
        late = [max_seq_len * 0.75, max_seq_len]
        rounds = np.concatenate([early, late])

    else:
        raise ValueError(f"Unknown method: {method}")

    # Convert to integers, ensure valid range, remove duplicates
    rounds = [int(round(r)) for r in rounds]
    rounds = [max(1, min(r, max_seq_len)) for r in rounds]
    rounds = sorted(set(rounds))

    # Always include 1 and max_seq_len
    if 1 not in rounds:
        rounds = [1] + rounds
    if max_seq_len not in rounds:
        rounds = rounds + [max_seq_len]

    return rounds

#key_rounds = get_key_rounds(max_seq_len, method=method, n_points=n_points)
#print(f"📊 Evaluating rounds: {key_rounds}")
#print(f"   Total: {len(key_rounds)} rounds instead of {max_seq_len}")

#ML Modules

### Load

In [14]:
def load_raw_datasets(config):


    if "ML" in config and "Events" in config["ML"]:
        events_cfg = config["ML"]["Events"]
    else:
        events_cfg = config["Events"]

    base = events_cfg["base_path"]
    files = events_cfg["files"]

    # --- Load all datasets ---
    df_voc = pd.read_csv(os.path.join(base, files["voc"]))
    df_sms = pd.read_csv(os.path.join(base, files["sms"]))
    df_app = pd.read_csv(os.path.join(base, files["app"]))
    df_user = pd.read_csv(os.path.join(base, files["user"]))

    # --- Normalize timestamps and add source column ---
    for df, src in [(df_voc, "VOC"), (df_sms, "SMS"), (df_app, "APP"), (df_user, "USER")]:
        df["source"] = src
        ts_col = [c for c in df.columns if "time" in c.lower()][0]
        df.rename(columns={ts_col: "event_time"}, inplace=True)
        df["event_time"] = pd.to_datetime(df["event_time"], errors="coerce")

    print("✅ Raw datasets loaded and timestamp-normalized.")
    return df_voc, df_sms, df_app, df_user

df_voc, df_sms, df_app, df_user = load_raw_datasets(config)


✅ Raw datasets loaded and timestamp-normalized.


### Build timeline (events)

In [15]:
def merge_and_prepare_events(df_voc, df_sms, df_app, df_user):

    # --- 1️⃣ Normalize USER dataset ---
    if 'label' not in df_user.columns:
        raise KeyError("❌ 'label' column not found in user dataset")

    # Ensure numeric consistency
    df_user['label'] = df_user['label'].fillna(0).astype(int)
    df_user['idcard_cnt'] = df_user['idcard_cnt'].fillna(0).astype(float)
    df_user['arpu_value'] = df_user['arpu_value'].fillna(0).astype(float)

    # --- 2️⃣ Extract static info for merging (label + sim count only) ---
    #static_user_info = df_user.groupby("phone_no_m", as_index=False)[["label", "idcard_cnt"]].max()
    # --- 2️⃣ Extract static info from the RAW user table (covers all 6,106 users) ---
    lbl_cfg = config["ML"]["labels"]
    raw_user = pd.read_csv(os.path.join(lbl_cfg["base_path"], lbl_cfg["file"]))
    static_user_info = raw_user[["phone_no_m", "label", "idcard_cnt"]].drop_duplicates("phone_no_m")
    static_user_info["label"] = static_user_info["label"].astype(int)
    static_user_info["idcard_cnt"] = static_user_info["idcard_cnt"].fillna(0).astype(float)

    # --- 3️⃣ Merge static info into other event tables ---
    df_voc = df_voc.merge(static_user_info, on="phone_no_m", how="left")
    df_sms = df_sms.merge(static_user_info, on="phone_no_m", how="left")
    df_app = df_app.merge(static_user_info, on="phone_no_m", how="left")


    # --- 4️⃣ Combine all transactional event sources ---
    # include df_user itself since arpu_value is event-like
    events = pd.concat([df_voc, df_sms, df_app, df_user], ignore_index=True)

    # --- 5️⃣ Fill and order ---
    #events["label"] = events["label"].fillna(0).astype(int)
    missing = int(events["label"].isna().sum())
    assert missing == 0, f"{missing} events have no label — label merge is broken"
    events["label"] = events["label"].astype(int)

    events["event_time"] = pd.to_datetime(events["event_time"], errors="coerce")
    events = events.sort_values(["phone_no_m", "event_time"]).reset_index(drop=True)

    # --- 6️⃣ Summary ---
    print("\n🔎 Feature Summary per Source:")
    for src, df in [("VOC", df_voc), ("SMS", df_sms), ("APP", df_app), ("USER", df_user)]:
        print(f"\n📂 Source: {src}")
        print(f"   Events: {len(df):,}")
        print(f"   Users : {df['phone_no_m'].nunique():,}")
        print(f"   Columns ({len(df.columns)}): {', '.join(df.columns)}")

    print("\n📊 Combined Dataset Summary:")
    print(f"   Total events: {len(events):,}")
    print(f"   Unique users: {events['phone_no_m'].nunique():,}")
    print(f"   Fraud ratio: {events['label'].mean()*100:.2f}%")
    user_labels = events.groupby("phone_no_m")["label"].max()
    print(f"   Fraud users: {int(user_labels.sum()):,} / {user_labels.size:,} ({user_labels.mean()*100:.2f}%)")

    return events

events = merge_and_prepare_events(df_voc, df_sms, df_app, df_user)

display(events.head())


🔎 Feature Summary per Source:

📂 Source: VOC
   Events: 5,015,430
   Users : 6,025
   Columns (11): phone_no_m, opposite_no_m, calltype_id, event_time, call_dur, city_name, county_name, imei_m, source, label, idcard_cnt

📂 Source: SMS
   Events: 6,848,509
   Users : 6,103
   Columns (7): phone_no_m, opposite_no_m, calltype_id, event_time, source, label, idcard_cnt

📂 Source: APP
   Events: 3,283,602
   Users : 6,106
   Columns (10): phone_no_m, event_time, source, busi_name, flow, month_id, flow_norm, month_str, label, idcard_cnt

📂 Source: USER
   Events: 39,454
   Users : 5,929
   Columns (10): phone_no_m, event_time, source, month_id, arpu_value, city_name, county_name, idcard_cnt, label, month_col

📊 Combined Dataset Summary:
   Total events: 15,186,995
   Unique users: 6,106
   Fraud ratio: 24.63%
   Fraud users: 1,962 / 6,106 (32.13%)


,phone_no_m,opposite_no_m,calltype_id,event_time,call_dur,city_name,county_name,imei_m,source,label,idcard_cnt,busi_name,flow,month_id,flow_norm,month_str,arpu_value,month_col
0,00073ceecc0f7220a440580ac5dea410c90d14b6669458...,df22edbc0e3dd6bc4f2f453e687b743e8442a54834b64f...,2.0,2019-08-12 08:09:11,NaN,NaN,NaN,NaN,SMS,0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,00073ceecc0f7220a440580ac5dea410c90d14b6669458...,df22edbc0e3dd6bc4f2f453e687b743e8442a54834b64f...,2.0,2019-08-12 08:09:11,NaN,NaN,NaN,NaN,SMS,0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,00073ceecc0f7220a440580ac5dea410c90d14b6669458...,df22edbc0e3dd6bc4f2f453e687b743e8442a54834b64f...,2.0,2019-08-12 08:09:11,NaN,NaN,NaN,NaN,SMS,0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,00073ceecc0f7220a440580ac5dea410c90d14b6669458...,df22edbc0e3dd6bc4f2f453e687b743e8442a54834b64f...,2.0,2019-08-13 16:21:53,NaN,NaN,NaN,NaN,SMS,0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,00073ceecc0f7220a440580ac5dea410c90d14b6669458...,df22edbc0e3dd6bc4f2f453e687b743e8442a54834b64f...,2.0,2019-08-13 16:21:53,NaN,NaN,NaN,NaN,SMS,0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Split data based on users (fraud, not fraud)

In [16]:


# ======================================
# Clean Numeric Columns
# ======================================
events = events.copy()
numeric_cols = events.select_dtypes(include=["number"]).columns.difference(["label"])

# Replace NaN with 0 for numeric fields (avoids scaling issues)
events[numeric_cols] = events[numeric_cols].fillna(0)

print(f"\n📊 Numeric columns to scale ({len(numeric_cols)}): {numeric_cols.tolist()}")


# ======================================
# 2 Create / Load Shared Train-Val-Test User Split
# ======================================
split_dir = os.path.join(config["root_path"], "splits", "shared_user_split_v1")
train_split_file = f"{split_dir}/train_users.csv"
test_split_file  = f"{split_dir}/test_users.csv"
val_split_file   = f"{split_dir}/val_users.csv"

os.makedirs(split_dir, exist_ok=True)

all_split_files_exist = (
    os.path.exists(train_split_file)
    and os.path.exists(test_split_file)
    and os.path.exists(val_split_file)
)

if all_split_files_exist:
    print("📂 Using shared user split files...")
    train_users = set(pd.read_csv(train_split_file)["phone_no_m"])
    test_users  = set(pd.read_csv(test_split_file)["phone_no_m"])
    val_users   = set(pd.read_csv(val_split_file)["phone_no_m"])
else:
    print("🆕 Creating shared train/test/val user split...")

    # One label per user
    user_labels = events.groupby("phone_no_m")["label"].max()
    fraud_users = user_labels[user_labels == 1].index
    normal_users = user_labels[user_labels == 0].index

    # 1) Train/Test split (split fraud user %)
    fraud_train, fraud_test = train_test_split(
        fraud_users, test_size=0.2, random_state=42
    )
    normal_train, normal_test = train_test_split(
        normal_users, test_size=0.2, random_state=42
    )

    train_users_full = set(fraud_train) | set(normal_train)
    test_users = set(fraud_test) | set(normal_test)

    # 2) Validation split from training users only
    train_user_labels = user_labels.loc[list(train_users_full)]

    fraud_train_users = train_user_labels[train_user_labels == 1].index
    normal_train_users = train_user_labels[train_user_labels == 0].index

    fraud_tr, fraud_val = train_test_split(
        fraud_train_users, test_size=0.2, random_state=42
    )
    normal_tr, normal_val = train_test_split(
        normal_train_users, test_size=0.2, random_state=42
    )

    train_users = set(fraud_tr) | set(normal_tr)
    val_users   = set(fraud_val) | set(normal_val)

    pd.DataFrame({"phone_no_m": sorted(train_users)}).to_csv(train_split_file, index=False)
    pd.DataFrame({"phone_no_m": sorted(test_users)}).to_csv(test_split_file, index=False)
    pd.DataFrame({"phone_no_m": sorted(val_users)}).to_csv(val_split_file, index=False)

    print(f"✅ Saved shared split files to '{split_dir}/'")
# ======================================
# Split overlap checks
# ======================================
assert len(train_users & test_users) == 0, "❌ User leakage: train and test overlap"
assert len(train_users & val_users) == 0, "❌ User leakage: train and val overlap"
assert len(test_users & val_users) == 0, "❌ User leakage: test and val overlap"

print("🔒 Split overlap checks passed:")
print(f"   train ∩ test = {len(train_users & test_users)}")
print(f"   train ∩ val  = {len(train_users & val_users)}")
print(f"   test  ∩ val  = {len(test_users & val_users)}")
user_labels = events.groupby("phone_no_m")["label"].max()
print(f"   sizes  train/val/test = {len(train_users)} / {len(val_users)} / {len(test_users)}")
print(f"   fraud  train/val/test = {int(user_labels.loc[list(train_users)].sum())} / "
      f"{int(user_labels.loc[list(val_users)].sum())} / {int(user_labels.loc[list(test_users)].sum())}")
# ======================================
# Apply Split to Events
# ======================================
train_events = events[events["phone_no_m"].isin(train_users)]
test_events  = events[events["phone_no_m"].isin(test_users)]
val_events = events[events["phone_no_m"].isin(val_users)]

# Sanity checks
assert len(set(train_events["phone_no_m"]) & set(test_events["phone_no_m"])) == 0, "❌ User leakage detected!"
assert train_events["label"].nunique() == 2, "❌ Training set must contain both classes"
assert test_events["label"].nunique() == 2, "❌ Test set must contain both classes"

# --- add time gap, scaled featur ---
# for name, df in [('train_events', train_events), ('test_events', test_events)]:
#     df = df.copy()  # avoid SettingWithCopyWarning
#     df['event_time'] = pd.to_datetime(df['event_time'])
#     #df.sort_values(['phone_no_m', 'event_time'], inplace=True)
#     df = df.sort_values(['phone_no_m', 'event_time'], kind='mergesort').reset_index(drop=True)
#     df['dt_hours'] = df.groupby('phone_no_m')['event_time'].diff().dt.total_seconds() / 3600
#     df['dt_hours'] = df['dt_hours'].fillna(0)
#     df['dt_hours'] = np.log1p(df['dt_hours'])  # normalize gaps
#     if name == 'train_events':
#         train_events = df
#     else:
#         test_events = df
for name, df in [
    ('train_events', train_events),
    ('val_events', val_events),
    ('test_events', test_events)
]:
    df = df.copy()
    df['event_time'] = pd.to_datetime(df['event_time'])
    df = df.sort_values(['phone_no_m', 'event_time'], kind='mergesort').reset_index(drop=True)
    df['dt_hours'] = df.groupby('phone_no_m')['event_time'].diff().dt.total_seconds() / 3600
    df['dt_hours'] = df['dt_hours'].fillna(0)
    df['dt_hours'] = np.log1p(df['dt_hours'])

    if name == 'train_events':
        train_events = df
    elif name == 'val_events':
        val_events = df
    else:
        test_events = df


# Store unscaled events BEFORE line 895
train_events_unscaled = train_events.copy()
test_events_unscaled = test_events.copy()
val_events_unscaled = val_events.copy()


# Sanity checks
assert len(set(train_events["phone_no_m"]) & set(test_events["phone_no_m"])) == 0, "❌ User leakage detected!"
assert train_events["label"].nunique() == 2, "❌ Training set must contain both classes"
assert test_events["label"].nunique() == 2, "❌ Test set must contain both classes"
scale_cols_seq = sorted(set(train_events.select_dtypes(include=["number"]).columns) - {"label"})
scaler_seq = StandardScaler().fit(train_events[scale_cols_seq])
train_events[scale_cols_seq] = scaler_seq.transform(train_events[scale_cols_seq])
test_events[scale_cols_seq]  = scaler_seq.transform(test_events[scale_cols_seq])
val_events[scale_cols_seq]   = scaler_seq.transform(val_events[scale_cols_seq])

# ======================================
# 4️⃣ snapshot
# ======================================

# ======================================
# 4️⃣ Create Sequences (using multiple features)
# ======================================
numeric_features = [c for c in numeric_cols if c not in ["label"]]  # exclude label
if 'dt_hours' in train_events.columns:
    numeric_features.append('dt_hours')
print(f"\n📦 Features used for sequences: {numeric_features}")
feature_cols_tf = numeric_features.copy()
# 👉 Transformer feature columns: same numeric features as LSTM + source_id

if "source_id" not in feature_cols_tf:
    feature_cols_tf.append("source_id")

print("[Transformer] feature_cols_tf:", feature_cols_tf)



📊 Numeric columns to scale (6): ['arpu_value', 'call_dur', 'calltype_id', 'flow', 'flow_norm', 'idcard_cnt']
📂 Using shared user split files...
🔒 Split overlap checks passed:
   train ∩ test = 0
   train ∩ val  = 0
   test  ∩ val  = 0
   sizes  train/val/test = 3907 / 977 / 1222
   fraud  train/val/test = 1255 / 314 / 393

📦 Features used for sequences: ['arpu_value', 'call_dur', 'calltype_id', 'flow', 'flow_norm', 'idcard_cnt', 'dt_hours']
[Transformer] feature_cols_tf: ['arpu_value', 'call_dur', 'calltype_id', 'flow', 'flow_norm', 'idcard_cnt', 'dt_hours', 'source_id']


### Generate_snapshots_from_events

In [17]:
# ============================================================
# 🔧 OPTIMIZED SNAPSHOT GENERATION FROM EVENTS (FIXED)
# ============================================================

def generate_snapshots_from_events(
    events_df,
    users,
    r,
    max_seq_len,
    recent_mode=True
):
    """
    Generate snapshot features from events DataFrame.
    OPTIMIZED: Uses groupby instead of per-user filtering.
    """

    # 1️⃣ Filter to relevant users FIRST (huge speedup)
    events_filtered = events_df[events_df["phone_no_m"].isin(users)].copy()

    if events_filtered.empty:
        return pd.DataFrame(), np.array([]), np.array([])

    # 2️⃣ Sort once
    events_filtered = events_filtered.sort_values(["phone_no_m", "event_time"])

    # 3️⃣ Apply selection logic per user using groupby
    def select_events(df_u):
        if recent_mode:
            df_u = df_u.tail(max_seq_len).head(r)
        else:
            df_u = df_u.head(r)
        return df_u

    selected = events_filtered.groupby("phone_no_m", group_keys=False).apply(select_events)

    if selected.empty:
        return pd.DataFrame(), np.array([]), np.array([])

    # 4️⃣ Aggregate features using groupby
    snapshot_rows = []

    for user, df_u in tqdm(selected.groupby("phone_no_m"), desc="Generating snapshots"):
    #for user, df_u in selected.groupby("phone_no_m"):
        row = {"phone_no_m": user}

        # Get label
        label = int(df_u["label"].max()) if "label" in df_u.columns else 0

        # --- Voice Features ---
        voc = df_u[df_u["source"] == "VOC"]
        if len(voc) > 0:
            if "call_dur" in voc.columns:
                call_dur = pd.to_numeric(voc["call_dur"], errors="coerce").fillna(0)
            else:
                call_dur = pd.Series([0])

            row["voc_total_calls"] = len(voc)
            row["voc_unique_contacts"] = voc["opposite_no_m"].nunique() if "opposite_no_m" in voc.columns else 0
            row["voc_total_duration"] = call_dur.sum()
            row["voc_avg_duration"] = call_dur.mean()
            row["voc_max_duration"] = call_dur.max()
            row["voc_std_duration"] = call_dur.std() if len(call_dur) > 1 else 0
            row["voc_active_days"] = voc["event_time"].dt.weekday.nunique()
            row["voc_active_hours"] = voc["event_time"].dt.hour.nunique()
        else:
            row.update({
                "voc_total_calls": 0, "voc_unique_contacts": 0, "voc_total_duration": 0,
                "voc_avg_duration": 0, "voc_max_duration": 0, "voc_std_duration": 0,
                "voc_active_days": 0, "voc_active_hours": 0
            })

        # --- SMS Features ---
        sms = df_u[df_u["source"] == "SMS"]
        if len(sms) > 0:
            row["sms_total_msgs"] = len(sms)
            row["sms_unique_contacts"] = sms["opposite_no_m"].nunique() if "opposite_no_m" in sms.columns else 0
            row["sms_active_hours"] = sms["event_time"].dt.hour.nunique()
            if "calltype_id" in sms.columns:
                calltype = pd.to_numeric(sms["calltype_id"], errors="coerce")
                row["sms_calltype_ratio"] = (calltype == 1).mean()
            else:
                row["sms_calltype_ratio"] = 0
        else:
            row.update({
                "sms_total_msgs": 0, "sms_unique_contacts": 0,
                "sms_active_hours": 0, "sms_calltype_ratio": 0
            })

        # --- App Features ---
        app = df_u[df_u["source"] == "APP"]
        if len(app) > 0:
            if "flow" in app.columns:
                flow = pd.to_numeric(app["flow"], errors="coerce").fillna(0)
            else:
                flow = pd.Series([0])

            row["app_months_active"] = app["month_id"].nunique() if "month_id" in app.columns else 0
            row["app_total_flow"] = flow.sum()
            row["app_avg_flow"] = flow.mean()
            row["app_std_flow"] = flow.std() if len(flow) > 1 else 0
            row["app_unique_apps_mean"] = app["busi_name"].nunique() if "busi_name" in app.columns else 0
            row["app_unique_apps_max"] = app["busi_name"].nunique() if "busi_name" in app.columns else 0
        else:
            row.update({
                "app_months_active": 0, "app_total_flow": 0, "app_avg_flow": 0,
                "app_std_flow": 0, "app_unique_apps_mean": 0, "app_unique_apps_max": 0
            })

        # --- User/ARPU Features ---
        arpu = df_u[df_u["source"] == "USER"]
        if len(arpu) > 0:
            if "arpu_value" in arpu.columns:
                arpu_val = pd.to_numeric(arpu["arpu_value"], errors="coerce")
            else:
                arpu_val = pd.Series([0])

            row["user_months_active"] = arpu["month_id"].nunique() if "month_id" in arpu.columns else 0
            row["arpu_mean"] = arpu_val.mean()
            row["arpu_std"] = arpu_val.std() if len(arpu_val) > 1 else 0
            row["arpu_max"] = arpu_val.max()
            row["idcard_cnt"] = arpu["idcard_cnt"].max() if "idcard_cnt" in arpu.columns else 0
        else:
            row.update({
                "user_months_active": 0, "arpu_mean": 0, "arpu_std": 0,
                "arpu_max": 0, "idcard_cnt": 0
            })

        # Meta features
        row["window_size"] = r
        row["snapshot_round"] = r
        row["label"] = label

        snapshot_rows.append(row)

    # Build DataFrame
    df_snapshot = pd.DataFrame(snapshot_rows).fillna(0)

    if df_snapshot.empty:
        return pd.DataFrame(), np.array([]), np.array([])

    y = df_snapshot["label"].values
    user_ids = df_snapshot["phone_no_m"].values
    X = df_snapshot.drop(columns=["phone_no_m", "label"])

    return X, y, user_ids



#  ▶ Classic Ml Snapshot based

# ▶  Advance ML

### make_user_sequences

In [18]:

# ============================================================
# 2️⃣ Selector functions (FIXED, SIMPLE)
# ============================================================

def selector_oldest(r):
    """Select oldest r events"""
    return lambda df_u: df_u.head(r)

def selector_last_r(r):
    """Select LAST r events (to match full evaluation behavior)"""
    return lambda df_u: df_u.tail(r)

def selector_most_recent(r):
    """Select most recent r events (used AFTER window freeze)"""
    return lambda df_u: df_u.tail(r)

def make_user_sequences(
    events,
    feature_cols=None,
    max_seq_len=100,
    event_selector=None,
):
    events = events.copy()
    users, X_seq, y = [], [], []

    SOURCE_MAP = {
        "APP": 0,
        "SMS": 1,
        "USER": 2,
        "VOC": 3,
    }

    unknown_sources = set(events["source"].astype(str).unique()) - set(SOURCE_MAP.keys())
    assert len(unknown_sources) == 0, f"❌ Unknown source values found: {unknown_sources}"

    events["source_id"] = events["source"].map(SOURCE_MAP).astype(int)

    if feature_cols is None:
        feature_cols = events.select_dtypes(include=["number"]) \
                             .columns.difference(["label"]) \
                             .tolist()
    if "source_id" not in feature_cols:
        feature_cols.append("source_id")

    for user, df_u in events.groupby("phone_no_m"):

        # 1️⃣ Sort
        df_u = df_u.sort_values("event_time")

        # 2️⃣ Freeze to last max_seq_len
        if recent_mode:
            df_u = df_u.tail(max_seq_len)   # E51..E100

        # 3️⃣ 🔁 APPLY PROGRESSIVE SELECTION HERE
        if event_selector is not None:
            df_u = event_selector(df_u)

        # 4️⃣ Features
        feats = df_u[feature_cols].to_numpy(dtype=float)

        # 5️⃣ Pad / truncate
        if len(feats) < max_seq_len:
            feats = np.pad(feats, ((max_seq_len - len(feats), 0), (0, 0)))
        else:
            feats = feats[-max_seq_len:]

        label = int(df_u["label"].max())

        X_seq.append(feats)
        y.append(label)
        users.append(user)

    return np.array(X_seq), np.array(y), np.array(users)



### Create sequences

In [19]:

trans_X_train, trans_y_train, users_train = make_user_sequences(train_events, feature_cols=numeric_features, max_seq_len=max_seq_len,event_selector=selector_oldest(max_seq_len))
trans_X_test, trans_y_test, users_test = make_user_sequences(test_events, feature_cols=numeric_features, max_seq_len=max_seq_len,event_selector=selector_oldest(max_seq_len))
trans_X_val, trans_y_val, _ =            make_user_sequences(val_events,    feature_cols=numeric_features,max_seq_len=max_seq_len, event_selector=selector_oldest(max_seq_len))
assert len(set(val_events["phone_no_m"])) > 0, "❌ Validation set is empty!"
assert val_events["label"].nunique() == 2, "❌ Validation set must contain both classes"

print("\n✅ Sequence Summary (per-user sequences):")
print(f"   X_train: {trans_X_train.shape} | Fraud ratio: {np.mean(trans_y_train)*100:.2f}%")
print(f"   X_test : {trans_X_test.shape} | Fraud ratio: {np.mean(trans_y_test)*100:.2f}%")

# ======================================
# 5️⃣ Consistency Check
# ======================================
trans_rf_train = set(pd.read_csv(train_split_file)["phone_no_m"])
trans_rf_test  = set(pd.read_csv(test_split_file)["phone_no_m"])
assert trans_rf_train == train_users, "❌ Train user mismatch between LSTM and RF/XGB!"
assert trans_rf_test  == test_users,  "❌ Test user mismatch between LSTM and RF/XGB!"
print("\n🔒 Consistency Check: ✅ Same users used for all models (LSTM, RF, XGBoost).")



✅ Sequence Summary (per-user sequences):
   X_train: (3907, 6, 8) | Fraud ratio: 32.12%
   X_test : (1222, 6, 8) | Fraud ratio: 32.16%

🔒 Consistency Check: ✅ Same users used for all models (LSTM, RF, XGBoost).


#Pretrained

#TimesNet

## Preparation

###Install

In [20]:

# ============================
# 1️⃣ Clean up any old copies
# ============================
#!rm -rf /content/drive/MyDrive/Sem-6/coding/github/fraud_detection/Time-Series-Library

# ============================
# 2️⃣ Clone directly from gethub
# ============================
#!git clone https://github.com/thuml/Time-Series-Library.git /content/drive/MyDrive/Sem-6/coding/github/fraud_detection/Time-Series-Library

# ============================
# 3️⃣ Add repo to Python path
# ============================
import sys
sys.path.append('/content/drive/MyDrive/Sem-6/coding/github/fraud_detection/Time-Series-Library')

print("✅ Basic environment ready for TSLib!")

# ============================
# 4️⃣ Verify repo structure
# ============================
%cd /content/drive/MyDrive/Sem-6/coding/github/fraud_detection/Time-Series-Library
!ls -lh run.py




✅ Basic environment ready for TSLib!
/content/drive/MyDrive/Sem-6/coding/github/fraud_detection/Time-Series-Library
-rw------- 1 root root 16K Apr 29 16:31 run.py


### Evaluate experiment

In [21]:

def evaluate_experiment(
    results_dir: str,
    threshold,
    num_classes: int = None,
    class_labels=None,
    title: str = None,
    show_plot: bool = True,
):
    pred_path = os.path.join(results_dir, "pred.npy")
    true_path = os.path.join(results_dir, "true.npy")
    prob_path = os.path.join(results_dir, "prob.npy")

    if not os.path.exists(pred_path) or not os.path.exists(true_path):
        raise FileNotFoundError(f"❌ Could not find pred.npy or true.npy in {results_dir}")

    true = np.load(true_path).flatten()

    # ── Probabilities available → use passed validation threshold ──
    if os.path.exists(prob_path):
        probs = np.load(prob_path).flatten()

        try:
            auc_val = roc_auc_score(true, probs) if len(np.unique(true)) == 2 else np.nan
        except:
            auc_val = np.nan

        pred = (probs >= threshold).astype(int)
        best_thr = threshold

    # ── Fallback → hard labels only ──
    else:
        pred = np.load(pred_path).flatten()
        best_thr = 0.5
        try:
            auc_val = roc_auc_score(true, pred) if len(np.unique(true)) == 2 else np.nan
        except:
            auc_val = np.nan

    # ── Metrics from final pred ──
    acc  = accuracy_score(true, pred)
    prec = precision_score(true, pred, average="binary", zero_division=0)
    rec  = recall_score(true, pred, average="binary", zero_division=0)
    f1   = f1_score(true, pred, average="binary", zero_division=0)

    # ── Confusion matrix ──
    cm = confusion_matrix(true, pred)
    if class_labels is None:
        class_labels = [str(c) for c in np.unique(true)]

    # ── Plot ──
    if show_plot:
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_labels)
        plt.figure(figsize=(8, 8))
        disp.plot(cmap="Blues", values_format="d", colorbar=False)
        plt.title(title or f"Confusion Matrix ({os.path.basename(results_dir)})")
        plt.show()

    # ── Print ──
    print(f"\n📊 Accuracy: {acc:.4f}")
    print(f"📈 Precision: {prec:.4f} | Recall: {rec:.4f} | F1: {f1:.4f} | AUC: {auc_val:.4f}")
    print(f"🎯 Threshold: {best_thr:.3f}")
    print("\nDetailed Report:")
    print(classification_report(true, pred, target_names=class_labels, digits=4))

    return {
        "NAS SL": max_seq_len,
        "Testing SL": max_seq_len,
        "f1": f1,
        "accuracy": acc,
        "recall": rec,
        "precision": prec,
        "auc": auc_val,
        "confusion_matrix": cm,
        "threshold": round(best_thr, 3),
    }

### Data handling  

#### Covert  data to ts format

In [22]:


def write_ts_file(
    X, y, split_name,
    problem_name="FraudDataset",
    out_dir=None,
    pad_to_dim=None,
    round_id=None
):


    if out_dir is None:
        out_dir = f"/content/drive/MyDrive/Sem-6/coding/github/fraud_detection/ExternalDataset/TimesNet/classification/FraudDataset_NAS_{max_seq_len}"
    os.makedirs(out_dir, exist_ok=True)
    suffix = f"_R{round_id}" if round_id is not None else ""
    ts_path = os.path.join(
        out_dir,
        f"{problem_name}_{split_name.upper()}{suffix}.ts"
    )


    with open(ts_path, "w") as f:
        f.write(f"@problemName {problem_name}\n")
        f.write("@timeStamps false\n")
        f.write("@univariate false\n")
        f.write("@classLabel true 0 1\n")
        f.write("@data\n")

        for i in range(len(X)):
            arr = np.array(X[i])

            # --- Pad or trim to target feature dimension ---
            if pad_to_dim is not None:
                if arr.ndim == 1:
                    arr = arr.reshape(-1, 1)
                n_dim = arr.shape[1]
                if n_dim < pad_to_dim:
                    pad = np.zeros((arr.shape[0], pad_to_dim - n_dim))
                    arr = np.hstack((arr, pad))
                elif n_dim > pad_to_dim:
                    arr = arr[:, :pad_to_dim]

            # --- Convert to string format ---
            if arr.ndim == 1:
                arr_str = ",".join(map(str, arr))
            else:
                parts = [",".join(map(str, arr[:, d])) for d in range(arr.shape[1])]
                arr_str = " : ".join(parts)  # colon separates dimensions

            f.write(f"{arr_str}:{int(y[i])}\n")

    print(f"✅ Wrote {ts_path} with {len(X)} samples (strict sktime format)")
    if pad_to_dim:
        print(f"📏 Feature dimensions padded/trimmed to {pad_to_dim}")


#### Convert

In [23]:

write_ts_file(trans_X_train, trans_y_train, split_name="TRAIN", pad_to_dim=13)
write_ts_file(trans_X_test, trans_y_test, split_name="TEST", pad_to_dim=13)
write_ts_file(trans_X_val, trans_y_val, split_name="VAL", pad_to_dim=13)


# write_ts_file(Xtr_raw, ytr, split_name="TRAIN")
# write_ts_file(Xte_raw, yte, split_name="TEST")

!ls -lh /content/drive/MyDrive/Sem-6/coding/github/fraud_detection/ExternalDataset/TimesNet/classification/FraudDataset_NAS_{max_seq_len}/

file_path = f"/content/drive/MyDrive/Sem-6/coding/github/fraud_detection/ExternalDataset/TimesNet/classification/FraudDataset_NAS_{max_seq_len}/FraudDataset_TRAIN.ts"
X, y = load_from_tsfile_to_dataframe(file_path)

print("✅ Loaded OK!")
print(X.shape)
print(set(y))


✅ Wrote /content/drive/MyDrive/Sem-6/coding/github/fraud_detection/ExternalDataset/TimesNet/classification/FraudDataset_NAS_6/FraudDataset_TRAIN.ts with 3907 samples (strict sktime format)
📏 Feature dimensions padded/trimmed to 13
✅ Wrote /content/drive/MyDrive/Sem-6/coding/github/fraud_detection/ExternalDataset/TimesNet/classification/FraudDataset_NAS_6/FraudDataset_TEST.ts with 1222 samples (strict sktime format)
📏 Feature dimensions padded/trimmed to 13
✅ Wrote /content/drive/MyDrive/Sem-6/coding/github/fraud_detection/ExternalDataset/TimesNet/classification/FraudDataset_NAS_6/FraudDataset_VAL.ts with 977 samples (strict sktime format)
📏 Feature dimensions padded/trimmed to 13
total 6.1M
-rw------- 1 root root 1.3M Jul  9 01:46 FraudDataset_TEST.ts
-rw------- 1 root root 3.9M Jul  9 01:46 FraudDataset_TRAIN.ts
-rw------- 1 root root 984K Jul  9 01:46 FraudDataset_VAL.ts
✅ Loaded OK!
(3907, 13)
{np.str_('1'), np.str_('0')}


#### Show sample

In [24]:
ts_dir = f"/content/drive/MyDrive/Sem-6/coding/github/fraud_detection/ExternalDataset/TimesNet/classification/FraudDataset_NAS_{max_seq_len}"

for fname in ["FraudDataset_TRAIN.ts", "FraudDataset_TEST.ts"]:
    fpath = os.path.join(ts_dir, fname)
    print(f" {fname}: exists = {os.path.exists(fpath)} | size = {os.path.getsize(fpath)/1024:.1f} KB")
    if os.path.exists(fpath):
        with open(fpath) as f:
            for i, line in enumerate(f):
                if i < 10:
                    print(line.strip())
                else:
                    break
        print("------\n")


 FraudDataset_TRAIN.ts: exists = True | size = 3934.7 KB
@problemName FraudDataset
@timeStamps false
@univariate false
@classLabel true 0 1
@data
-0.029840919093863435,-0.029840919093863435,-0.029840919093863435,-0.029840919093863435,-0.029840919093863435,-0.029840919093863435 : -0.23359317309003816,-0.23359317309003816,-0.23359317309003816,-0.23359317309003816,-0.23359317309003816,-0.23359317309003816 : 0.8381382030441827,0.8381382030441827,0.8381382030441827,0.8381382030441827,0.8381382030441827,0.8381382030441827 : -0.041480529072158906,-0.041480529072158906,-0.041480529072158906,-0.041480529072158906,-0.041480529072158906,-0.041480529072158906 : -0.2519196987865733,-0.2519196987865733,-0.2519196987865733,-0.2519196987865733,-0.2519196987865733,-0.2519196987865733 : -0.8170257391209248,-0.8170257391209248,-0.8170257391209248,-0.8170257391209248,-0.8170257391209248,-0.8170257391209248 : -0.5335819405272189,-0.5335819405272189,-0.5335819405272189,3.8769679998396516,-0.5335819405272189

#### Summrize

In [25]:

file_path = f"/content/drive/MyDrive/Sem-6/coding/github/fraud_detection/ExternalDataset/TimesNet/classification/FraudDataset_NAS_{max_seq_len}/FraudDataset_TRAIN.ts"

X, y = load_from_tsfile_to_dataframe(file_path)

print("✅ File loaded successfully!")
print(f"Shape of X: {X.shape}")
print(f"Number of labels: {len(set(y))}")
print(f"Labels: {set(y)}")
#print(f"Example (first row):\n", X.iloc[0])

print("Feature ranges per column:")
for col in range(X.shape[1]):
    values = np.concatenate(X.iloc[:, col].values)
    print(f"Column {col}: mean={np.mean(values):.3f}, std={np.std(values):.3f}")


✅ File loaded successfully!
Shape of X: (3907, 13)
Number of labels: 2
Labels: {np.str_('1'), np.str_('0')}
Feature ranges per column:
Column 0: mean=-0.012, std=0.363
Column 1: mean=-0.022, std=0.949
Column 2: mean=0.588, std=0.568
Column 3: mean=-0.041, std=0.001
Column 4: mean=-0.250, std=0.060
Column 5: mean=0.114, std=1.174
Column 6: mean=0.253, std=1.550
Column 7: mean=1.513, std=0.890
Column 8: mean=0.000, std=0.000
Column 9: mean=0.000, std=0.000
Column 10: mean=0.000, std=0.000
Column 11: mean=0.000, std=0.000
Column 12: mean=0.000, std=0.000


### NAS TimeNet full Training

In [26]:

# ============================================================
# NAS TIMESNET – FULL UNFROZEN TRAINING (ARCH SEARCH)
# ============================================================

import os
import shutil
import subprocess
import optuna
from optuna.samplers import TPESampler
import numpy as np
import pandas as pd



# ============================================================
# CONFIG (MATCHES YOUR PIPELINE)
# ============================================================
RUN_PY = "/content/drive/MyDrive/Sem-6/coding/github/fraud_detection/Time-Series-Library/run.py"
DATA_ROOT = f"/content/drive/MyDrive/Sem-6/coding/github/fraud_detection/ExternalDataset/TimesNet/classification/FraudDataset_NAS_{max_seq_len}/"
TEST_RESULTS_ROOT = f"./test_results/"


# ============================================================
# SAFETY UTIL
# ============================================================
def find_latest_results_dir(base=f"./test_results/"):
    if not os.path.exists(base):
        raise RuntimeError("❌ test_results directory does not exist")

    candidates = [
        os.path.join(base, d)
        for d in os.listdir(base)
        if os.path.isdir(os.path.join(base, d))
    ]
    if not candidates:
        raise RuntimeError("❌ No TimesNet results directory found")

    return max(candidates, key=os.path.getmtime)

# ============================================================
# RUN ONE NAS TRIAL (TRAIN + FULL EVAL)
# ============================================================

nas_trial_log = []
best_f1_so_far = -1.0
best_trial_so_far = -1
ENQUEUED_TRIAL_IDS = {0, 1, 2, 3,4,5}

def run_timesnet_nas(trial_id, params):

    model_id = f"NAS_TimesNet_{trial_id}"

    # ✅ FIXED: Use ALL parameters from search space
    cmd = f"""CUDA_VISIBLE_DEVICES=0 python -u {RUN_PY} \
    --task_name classification \
    --is_training 1 \
    --mode UnfrozenOptWL \
    --root_path {DATA_ROOT} \
    --model_id {model_id} \
    --model TimesNet \
    --data UEA \
    --data_path FraudDataset \
    --features M \
    --seq_len {max_seq_len} \
    --target OT \
    --gpu 0 \
    --use_gpu 1 \
    --patience {params["patience"]} \
    --batch_size {params["batch_size"]} \
    --train_epochs {epochs} \
    --learning_rate {params["lr"]} \
    --e_layers {params["e_layers"]} \
    --d_model {params["d_model"]} \
    --d_ff {params["d_ff"]} \
    --top_k {params["top_k"]} \
    --num_kernels {params["num_kernels"]} \
    --dropout {params["dropout"]} \
    --des NAS_UnfrozenOptWL_Search \
    --itr 1
    """
    # Note: weight_decay removed since run.py may not support it

    print(cmd)
    get_ipython().system(cmd)

    print("==========================xxxxxxxxxxxxxxxxxx===============")

    # --------------------------------------------------------
    # Locate results safely (NO guessing)
    # --------------------------------------------------------
    #results_dir = find_latest_results_dir()
    # ✅ FIX — read from exact folder matching current trial
    setting = (
        f"classification_{model_id}_TimesNet_UEA_ftM_"
        f"sl{max_seq_len}_ll48_pl0_"
        f"dm{params['d_model']}_nh8_"
        f"el{params['e_layers']}_dl1_"
        f"df{params['d_ff']}_expand2_dc4_fc1_"
        f"ebtimeF_dtTrue_NAS_UnfrozenOptWL_Search_0"
    )
    results_dir = os.path.join(f"./test_results/", setting)

    val_true_path = os.path.join(results_dir, "val_true.npy")
    val_prob_path  = os.path.join(results_dir, "val_prob.npy")
    if not os.path.exists(val_prob_path) or not os.path.exists(val_true_path):
        raise RuntimeError("❌ val_prob.npy / val_true.npy not found")
    val_probs = np.load(val_prob_path).flatten()
    val_true  = np.load(val_true_path).flatten()

    best_val_f1 = -1.0
    best_val_thr = 0.5

    for thr in np.linspace(0.2, 0.8, 61):
        f1 = f1_score(val_true, (val_probs >= thr).astype(int), zero_division=0)
        if f1 > best_val_f1:
            best_val_f1 = f1
            best_val_thr = thr
# --- VALIDATION metrics using best validation threshold ---
    val_preds = (val_probs >= best_val_thr).astype(int)

    val_auc = roc_auc_score(val_true, val_probs)
    val_recall = recall_score(val_true, val_preds, zero_division=0)
    val_precision = precision_score(val_true, val_preds, zero_division=0)

    # --- TEST F1 ---
    test_true_path = os.path.join(results_dir, "true.npy")
    test_prob_path  = os.path.join(results_dir, "prob.npy")
    if not os.path.exists(test_prob_path) or not os.path.exists(test_true_path):
        raise RuntimeError("❌ prob.npy / true.npy not found")
    test_probs = np.load(test_prob_path).flatten()
    test_true  = np.load(test_true_path).flatten()

    best_test_f1 = -1.0
    best_thr = 0.5
    for thr in np.linspace(0.2, 0.8, 61):
        f1 = f1_score(test_true, (test_probs >= thr).astype(int), zero_division=0)
        if f1 > best_test_f1:
            best_test_f1 = f1
            best_thr = thr
# --- TEST metrics using validation threshold (scientifically correct) ---
    test_preds = (test_probs >= best_val_thr).astype(int)

    test_auc = roc_auc_score(test_true, test_probs)
    test_recall = recall_score(test_true, test_preds, zero_division=0)
    test_precision = precision_score(test_true, test_preds, zero_division=0)
    test_f1 = f1_score(test_true, test_preds, zero_division=0)

    # ========================================================
    # Trial summary row
    # ========================================================
    now = datetime.now()
    global best_f1_so_far, best_trial_so_far

    if best_val_f1 > best_f1_so_far:
        best_f1_so_far = best_val_f1
        best_trial_so_far = trial_id

    is_enqueued_trial = trial_id in ENQUEUED_TRIAL_IDS

    trial_row = {
      "date": now.strftime("%Y-%m-%d"),
      "time": now.strftime("%H:%M:%S"),
      "seq_length": max_seq_len,
      "trial_id": trial_id,

      # 🔹 Validation (optimization target)
      "val_f1": round(best_val_f1, 4),
      "val_auc": round(val_auc, 4),
      "val_recall": round(val_recall, 4),
      "val_precision": round(val_precision, 4),

      # 🔹 Model parameters
      "d_model": params["d_model"],
      "lr": params["lr"],
      "d_ff": params["d_ff"],
      "e_layers": params["e_layers"],
      "top_k": params["top_k"],
      "num_kernels": params["num_kernels"],
      "dropout": params["dropout"],
      "batch_size": params["batch_size"],
      "patience": params["patience"],

      # 🔹 Thresholds
      "val_threshold": round(best_val_thr, 3),
      "best_test_threshold": round(best_thr, 3),

      # 🔹 Best tracking
      "best_val_so_far": round(best_f1_so_far, 4),
      "best_trial_id": best_trial_so_far,

      # 🔹 Test (monitoring only)
      "test_f1": round(test_f1, 4),
      "test_auc": round(test_auc, 4),
      "test_recall": round(test_recall, 4),
      "test_precision": round(test_precision, 4),

      # 🔹 Diagnostics
      "gap_val_test": round(best_val_f1 - test_f1, 4),
      "is_enqueued": is_enqueued_trial,
      "status": "OK",
  }

    nas_trial_log.append(trial_row)
    display(
    pd.DataFrame([trial_row])
    .sort_values("val_f1", ascending=False)
    .reset_index(drop=True)
)

#     display(
#     pd.DataFrame(nas_trial_log)
#     .sort_values("val_f1", ascending=False)
#     .reset_index(drop=True)
# )
    return best_val_f1


# ============================================================
# OPTUNA OBJECTIVE
# ============================================================
def objective(trial):
    params = {
        "lr": trial.suggest_float("lr", 5e-5, 3e-3, log=True),
        "d_model": trial.suggest_categorical("d_model", [2, 4, 8, 16, 32, 48, 64, 128, 256]),
        "d_ff": trial.suggest_categorical("d_ff", [2, 4, 8, 16, 32, 64, 128, 256, 512]),
        "e_layers": trial.suggest_int("e_layers", 2, 6),
        "top_k": trial.suggest_categorical("top_k", [2, 3]),
        "dropout": trial.suggest_float("dropout", 0.0, 0.4),
        "batch_size": trial.suggest_categorical("batch_size", [8, 16]),
        "patience": trial.suggest_int("patience", 2, 5),
        "num_kernels": trial.suggest_categorical("num_kernels", [3, 4, 6]),
    }

    try:
        return run_timesnet_nas(trial.number, params)
    except Exception as e:
        print(f"❌ Trial {trial.number} failed: {e}")
        raise optuna.exceptions.TrialPruned(str(e))


# ============================================================
# RUN NAS SEARCH
# ============================================================

# ✅ Better sampler for smarter search
sampler = TPESampler(
    n_startup_trials=10,      # Random trials before TPE kicks in
    n_ei_candidates=24,       # More candidates for acquisition function
    multivariate=True,        # Consider parameter correlations
    seed=42
)

study = optuna.create_study(
    direction="maximize",
    sampler=sampler
    #pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=3)
)


study.enqueue_trial({
    "lr": 1e-3,
    "d_model": 64,
    "d_ff": 128,
    "e_layers": 3,
    "top_k": 3,
    "dropout": 0.2,
    "batch_size": 8,
    "patience": 3
})
study.enqueue_trial({
    'lr': 0.000936,
    'd_model': 48,
    'd_ff': 4,
    'e_layers': 2,
    'top_k': 3,
    'dropout': 0.387,
    'batch_size': 8,
    'patience': 5
})
study.enqueue_trial({
    "lr": 1e-3,
    "d_model": 4,
    "d_ff": 2,
    "e_layers": 2,
    "top_k": 2,
    "dropout": 0.0,
    "batch_size": 8,
    "patience": 3,
    "num_kernels": 6

})

# Trial 2: dxs config (best F1)
study.enqueue_trial({
    "lr": 1e-3,
    "d_model": 4,
    "d_ff": 2,
    "e_layers": 2,
    "top_k": 2,
    "dropout": 0.3,
    "batch_size": 8,
    "patience": 3,
    "num_kernels": 6
})

# Trial 3: s config variant
study.enqueue_trial({
    "lr": 1e-4,
    "d_model": 32,
    "d_ff": 32,
    "e_layers": 2,
    "top_k": 2,
    "dropout": 0.1,
    "batch_size": 8,
    "patience": 3,
    "num_kernels": 6
})



# ✅ ADD: Very small model (to show minimum threshold)
study.enqueue_trial({
    "lr": 1e-3,
    "d_model": 2,
    "d_ff": 2,
    "e_layers": 2,
    "top_k": 2,
    "dropout": 0.0,
    "batch_size": 16,
    "patience": 3,
    "num_kernels": 6
})



max_attempts = n_trials + 20  # allow up to 20 failures
attempts = 0
while len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]) < n_trials:
    if attempts >= max_attempts:
        print(f"⚠️ Reached {max_attempts} attempts, stopping with {len(nas_trial_log)} valid trials")
        break
    study.optimize(objective, n_trials=1)
    attempts += 1

BEST_TRIAL = study.best_trial.number
BEST_PARAMS = study.best_trial.params
BEST_MODEL_ID = f"NAS_TimesNet_{BEST_TRIAL}"

print("\n" + "="*60)
print("🎉 BEST NAS MODEL FOUND")
print("="*60)
print("Model ID:", BEST_MODEL_ID)
print("Best F1:", study.best_value)
print("Best Params:", BEST_PARAMS)

# ============================================================
# ANALYSIS: SHOW ALL RESULTS FOR YOUR PROOF
# ============================================================
print("\n" + "="*60)
print("📊 ALL TRIALS RANKED BY F1")
print("="*60)

trials_df = study.trials_dataframe()
trials_df = trials_df.sort_values("value", ascending=False)

# Show relevant columns

display(
    pd.DataFrame(nas_trial_log)
    .sort_values("val_f1", ascending=False)
    .reset_index(drop=True)
)

# ============================================================
# ANALYSIS: MODEL SIZE vs PERFORMANCE
# ============================================================
print("\n" + "="*60)
print("📈 MODEL SIZE vs PERFORMANCE ANALYSIS")
print("="*60)

# Group by d_model and show average F1
if "params_d_model" in trials_df.columns:
    size_analysis = trials_df.groupby("params_d_model")["value"].agg(["mean", "max", "count"])
    size_analysis.columns = ["Avg_F1", "Max_F1", "Trials"]
    size_analysis = size_analysis.sort_index()
    print("\nPerformance by d_model:")
    print(size_analysis.to_string())

# ============================================================
# SAVE RESULTS TO CSV
# ============================================================
OUT_DIR = os.path.join(config["output"]["results_dir"], "NAS_v2/")
os.makedirs(OUT_DIR, exist_ok=True)
results_path = f"{OUT_DIR}nas_timesnet_results_WL{max_seq_len}.csv"
trials_df.to_csv(results_path, index=False)
pd.DataFrame(nas_trial_log).sort_values("val_f1", ascending=False).to_csv(f"{OUT_DIR}nas_timesnet_trial_log_WL{max_seq_len}.csv", index=False)
print(f"\n💾 Results saved to: {results_path}")

/tmp/ipykernel_1019/3283602907.py:246: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  sampler = TPESampler(
[I 2026-07-09 01:47:02,016] A new study created in memory with name: no-name-3a6accdd-0fa2-4c0a-baa9-44fe20fe7ff4


CUDA_VISIBLE_DEVICES=0 python -u /content/drive/MyDrive/Sem-6/coding/github/fraud_detection/Time-Series-Library/run.py     --task_name classification     --is_training 1     --mode UnfrozenOptWL     --root_path /content/drive/MyDrive/Sem-6/coding/github/fraud_detection/ExternalDataset/TimesNet/classification/FraudDataset_NAS_6/     --model_id NAS_TimesNet_0     --model TimesNet     --data UEA     --data_path FraudDataset     --features M     --seq_len 6     --target OT     --gpu 0     --use_gpu 1     --patience 3     --batch_size 8     --train_epochs 20     --learning_rate 0.001     --e_layers 3     --d_model 64     --d_ff 128     --top_k 3     --num_kernels 4     --dropout 0.2     --des NAS_UnfrozenOptWL_Search     --itr 1
    
Using GPU
Use UnfrozenOptWL
Use GPU: cuda:0
🔧 Building model and loading metadata...

========== Experiment Arguments ==========
activation               : gelu
alpha                    : 0.1
anomaly_ratio            : 0.25
augmentation_ratio       : 0
batch_si

,date,time,seq_length,trial_id,val_f1,val_auc,val_recall,val_precision,d_model,lr,d_ff,e_layers,top_k,num_kernels,dropout,batch_size,patience,val_threshold,best_test_threshold,best_val_so_far,best_trial_id,test_f1,test_auc,test_recall,test_precision,gap_val_test,is_enqueued,status
0,2026-07-09,01:52:07,6,0,0.7089,0.8459,0.6943,0.7243,64,0.001,128,3,3,4,0.2,8,3,0.41,0.45,0.7089,0,0.7037,0.8274,0.6768,0.7328,0.0052,True,OK


[I 2026-07-09 01:52:08,139] Trial 0 finished with value: 0.7089430894308943 and parameters: {'lr': 0.001, 'd_model': 64, 'd_ff': 128, 'e_layers': 3, 'top_k': 3, 'dropout': 0.2, 'batch_size': 8, 'patience': 3, 'num_kernels': 4}. Best is trial 0 with value: 0.7089430894308943.


CUDA_VISIBLE_DEVICES=0 python -u /content/drive/MyDrive/Sem-6/coding/github/fraud_detection/Time-Series-Library/run.py     --task_name classification     --is_training 1     --mode UnfrozenOptWL     --root_path /content/drive/MyDrive/Sem-6/coding/github/fraud_detection/ExternalDataset/TimesNet/classification/FraudDataset_NAS_6/     --model_id NAS_TimesNet_1     --model TimesNet     --data UEA     --data_path FraudDataset     --features M     --seq_len 6     --target OT     --gpu 0     --use_gpu 1     --patience 5     --batch_size 8     --train_epochs 20     --learning_rate 0.000936     --e_layers 2     --d_model 48     --d_ff 4     --top_k 3     --num_kernels 3     --dropout 0.387     --des NAS_UnfrozenOptWL_Search     --itr 1
    
Using GPU
Use UnfrozenOptWL
Use GPU: cuda:0
🔧 Building model and loading metadata...

========== Experiment Arguments ==========
activation               : gelu
alpha                    : 0.1
anomaly_ratio            : 0.25
augmentation_ratio       : 0
batch

...

,date,time,seq_length,trial_id,val_f1,val_auc,val_recall,val_precision,d_model,lr,d_ff,e_layers,top_k,num_kernels,dropout,batch_size,patience,val_threshold,best_test_threshold,best_val_so_far,best_trial_id,test_f1,test_auc,test_recall,test_precision,gap_val_test,is_enqueued,status
0,2026-07-09,08:21:48,6,99,0.7192,0.8466,0.6975,0.7424,64,0.001016,8,4,3,4,0.147039,8,5,0.43,0.32,0.7273,47,0.6806,0.8266,0.6641,0.6979,0.0386,False,OK


[I 2026-07-09 08:21:49,001] Trial 99 finished with value: 0.7192118226600985 and parameters: {'lr': 0.0010158561875261464, 'd_model': 64, 'd_ff': 8, 'e_layers': 4, 'top_k': 3, 'dropout': 0.14703909828907138, 'batch_size': 8, 'patience': 5, 'num_kernels': 4}. Best is trial 47 with value: 0.7272727272727273.



🎉 BEST NAS MODEL FOUND
Model ID: NAS_TimesNet_47
Best F1: 0.7272727272727273
Best Params: {'lr': 0.0023941831338809795, 'd_model': 64, 'd_ff': 8, 'e_layers': 2, 'top_k': 2, 'dropout': 0.3986424997690311, 'batch_size': 8, 'patience': 4, 'num_kernels': 4}

📊 ALL TRIALS RANKED BY F1


,date,time,seq_length,trial_id,val_f1,val_auc,val_recall,val_precision,d_model,lr,d_ff,e_layers,top_k,num_kernels,dropout,batch_size,patience,val_threshold,best_test_threshold,best_val_so_far,best_trial_id,test_f1,test_auc,test_recall,test_precision,gap_val_test,is_enqueued,status
0,2026-07-09,04:36:10,6,47,0.7273,0.8508,0.6879,0.7714,64,0.002394,8,2,2,4,0.398642,8,4,0.38,0.35,0.7273,47,0.6720,0.8233,0.6412,0.7059,0.0553,False,OK
1,2026-07-09,07:42:59,6,92,0.7212,0.8467,0.6879,0.7579,32,0.002349,64,5,3,3,0.317775,8,5,0.49,0.47,0.7273,47,0.6940,0.8310,0.6463,0.7493,0.0272,False,OK
2,2026-07-09,06:20:43,6,75,0.7206,0.8575,0.7229,0.7184,32,0.001741,64,5,2,3,0.316385,8,5,0.31,0.31,0.7273,47,0.7047,0.8305,0.6921,0.7177,0.0160,False,OK
3,2026-07-09,07:12:50,6,86,0.7197,0.8538,0.7197,0.7197,128,0.000965,8,5,3,3,0.357533,16,5,0.34,0.35,0.7273,47,0.6859,0.8289,0.6972,0.6749,0.0339,False,OK
4,2026-07-09,03:47:06,6,34,0.7193,0.8542,0.6815,0.7616,64,0.002248,8,3,3,4,0.225091,16,4,0.49,0.38,0.7193,34,0.6821,0.8351,0.6361,0.7353,0.0372,False,OK
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,2026-07-09,02:55:39,6,19,0.6788,0.8375,0.7134,0.6474,8,0.000366,8,3,3,3,0.338104,16,4,0.22,0.25,0.7157,12,0.6667,0.8219,0.7023,0.6345,0.0121,False,OK
96,2026-07-09,02:09:15,6,5,0.5632,0.7293,0.5605,0.5659,2,0.001000,2,2,2,6,0.000000,16,3,0.34,0.24,0.7089,0,0.5570,0.7417,0.5598,0.5542,0.0062,True,OK
97,2026-07-09,04:15:54,6,41,0.5537,0.6878,0.6401,0.4879,2,0.001265,8,3,3,4,0.230647,16,4,0.27,0.26,0.7193,34,0.5329,0.6824,0.6183,0.4682,0.0208,False,OK
98,2026-07-09,02:25:30,6,10,0.5302,0.6562,0.6433,0.4509,2,0.002148,128,2,3,4,0.135240,8,3,0.24,0.32,0.7089,0,0.5197,0.6569,0.6387,0.4380,0.0105,False,OK



📈 MODEL SIZE vs PERFORMANCE ANALYSIS

Performance by d_model:
                  Avg_F1    Max_F1  Trials
params_d_model                            
2               0.536426  0.563200       4
4               0.690470  0.693811       6
8               0.694152  0.702028       3
16              0.705801  0.713311       9
32              0.705035  0.721202      28
48              0.704308  0.713600       8
64              0.710825  0.727273      25
128             0.708207  0.719745      14
256             0.707192  0.710438       3

💾 Results saved to: /content/drive/MyDrive/Sem-6/coding/github/fraud_detection/results/NAS_v2/nas_timesnet_results_WL6.csv


#freeze

In [27]:
%pip freeze > "{project_path}requirement/freez/NASEnhancedPretraindMLModleAdvance-lock.txt"

In [28]:
end = time.time()
elapsed = end - start_time

hours = int(elapsed // 3600)
minutes = int((elapsed % 3600) // 60)
seconds = int(elapsed % 60)

print(f"Total Time = {hours}h {minutes}m {seconds}s")

Total Time = 6h 38m 50s
